# CUDA MoE — Optional A100 Experiments

Run all in your A100 runtime. This notebook embeds the validated v0.2.0 source and evidence.
It runs **quick-suite CUDA Graph comparisons first**, then the **larger synthetic model-shaped
preset** (H=4096, I=14336, E=8), and exports a ZIP containing every result and diagnostic.
The core 81 CPU / 39 GPU tests and eager quick suite were already passed in the recorded run
and are skipped here. Optional failures are recorded independently so one does not hide the other.
Graph capture may be unsupported for some variants. Only successful replay measurements count.
Recorded results: 24 graph replay checks passed (1.74× median graph-to-graph ratio),
and 6 larger-preset oracle comparisons passed (1.06× median eager ratio).
The original 3.17× eager result remains the primary small/medium-block claim.


In [ ]:
#@title Check the A100 runtime and extract the complete project
import base64, hashlib, importlib.util, io, os, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Attach the A100 GPU: Runtime > Change runtime type > A100 > Run all.")
major, minor = torch.cuda.get_device_capability()
if major < 8:
    raise RuntimeError("The BF16 full engine needs an Ampere-or-newer GPU. Use your A100 runtime.")
os.environ.setdefault("MAX_JOBS", "2")
os.environ.setdefault("TORCH_CUDA_ARCH_LIST", f"{major}.{minor}")
nvcc = shutil.which("nvcc")
if not nvcc:
    nvcc = next((str(path) for path in sorted(Path('/usr/local').glob('cuda*/bin/nvcc'), reverse=True)
                 if path.is_file()), None)
if nvcc:
    os.environ.setdefault("CUDA_HOME", str(Path(nvcc).parent.parent))
    os.environ['PATH'] = str(Path(nvcc).parent) + os.pathsep + os.environ['PATH']
missing = []
for module, requirement in [('ninja', 'ninja>=1.11'), ('pytest', 'pytest>=8'),
                            ('triton', 'triton==3.6.0')]:
    if importlib.util.find_spec(module) is None:
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('GPU:', torch.cuda.get_device_name())
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
import triton
print('Triton:', triton.__version__)
PAYLOAD = (
    '...'
    '9BcAAMxOCQAAAA=='
)
data = base64.b64decode(PAYLOAD)
assert hashlib.sha256(data).hexdigest() == '4b3781c6a14d26d6f4b971c38f638ad8fb6fc214673e17cea9479d55c28aa4df'
BASE = Path('/content') if Path('/content').is_dir() else Path.cwd()
PROJECT = BASE / 'cuda-moe-dispatch-v02'
if PROJECT.exists():
    # Preserve an earlier run if Run all is clicked again.
    import datetime
    PROJECT = BASE / ('cuda-moe-dispatch-v02-' + datetime.datetime.now().strftime('%H%M%S'))
PROJECT.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    for item in archive.infolist():
        destination = (PROJECT / item.filename).resolve()
        if not destination.is_relative_to(PROJECT.resolve()):
            raise RuntimeError('Invalid embedded archive path')
    archive.extractall(PROJECT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT), '--no-deps'])
print('Complete repository extracted:', PROJECT)
print('Setup complete. Keep this PyTorch install; the next cell compiles and validates the engine.')


## Run the full engine

The first build compiles CUDA sources and the grouped-matmul kernel. Compilation and validation
are excluded from benchmark timings. Correctness failures stop the comparison and export an error
bundle. Every speed ratio reports the measured included baseline; values below 1 mean a slowdown.


In [ ]:
#@title Compile, test, benchmark, show the report, and download everything
# Defaults finish the complete small/medium comparison first. Extras are switches.
RUN_CORE = False            # Reproduce CPU/GPU checks and smoke/quick eager results.
RUN_MODEL_SHAPED = True  # H=4096, I=14336, E=8; synthetic weights use several GB.
RUN_CUDA_GRAPHS = True   # Optional replay experiment; reported separately from eager forward.
RUN_PROFILE = False
EXPERT_BACKEND = 'triton'
from IPython.display import HTML, FileLink, display
import json, statistics, traceback
RUNS = PROJECT / 'runs'
RUNS.mkdir(exist_ok=True)
LOG = RUNS / 'run.log'


def run_command(*arguments):
    command = [sys.executable, '-u', *arguments]
    print('Running:', ' '.join(command), flush=True)
    with LOG.open('a') as log:
        log.write('\nCOMMAND: ' + ' '.join(command) + '\n')
        process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        code = process.wait()
    if code:
        raise RuntimeError(f'Command failed with exit {code}. Complete output is in runs/run.log.')


failure = None
try:
    run_command('-m', 'moe_dispatch', 'check-env')
    if RUN_CORE:
        run_command('-m', 'pytest', '-q', '-m', 'not gpu')
    if RUN_CORE:
        run_command('-m', 'pytest', '-q', '-m', 'gpu')
    if RUN_CORE:
        run_command('-m', 'moe_dispatch', 'validate')
    for suite in (('smoke', 'quick') if RUN_CORE else ()):
        run_command('-m', 'moe_dispatch', 'block-benchmark', '--suite', suite,
                    '--expert-backend', EXPERT_BACKEND, '--output', f'runs/block-{suite}')
    for enabled, suite, destination, flags in (
        (RUN_CUDA_GRAPHS, 'quick', 'runs/block-graphs', ('--cuda-graphs',)),
        (RUN_MODEL_SHAPED, 'model', 'runs/model-shaped', ()),
    ):
        if enabled:
            try:
                run_command('-m', 'moe_dispatch', 'block-benchmark', '--suite', suite,
                            '--expert-backend', EXPERT_BACKEND, '--output', destination, *flags)
                display(HTML((PROJECT / destination / 'block_report.html').read_text()))
            except RuntimeError as error:
                print('Optional experiment failed:', destination, error)
                (RUNS / (Path(destination).name + '_error.txt')).write_text(str(error))
    if RUN_PROFILE:
        try:
            run_command('-m', 'moe_dispatch', 'profile-block', '--expert-backend', EXPERT_BACKEND,
                        '--output', 'runs/moe_block_trace.json')
        except RuntimeError as error:
            print('Optional profiler unavailable:', error)
            (RUNS / 'profiler_error.txt').write_text(str(error))
    result_path = RUNS / 'block-quick/block_results.json'
    if not result_path.exists():
        result_path = PROJECT / 'benchmarks/a100-block/block_results.json'
    result = json.loads(result_path.read_text())
    forwards = {(row['case_id'], row['implementation']): row['median_us']
                for row in result['benchmarks'] if row['stage'] == 'forward'}
    ratios = [forwards[case['case_id'], 'torch'] / forwards[case['case_id'], 'cuda_fused']
              for case in result['configurations']]
    print('Validated full-forward configurations:', len(ratios))
    print('Actual median full-forward ratio:', round(statistics.median(ratios), 3), 'x')
    print('All case ratios:', [round(value, 3) for value in ratios])
    print('Includes router, dispatch, grouped experts, SwiGLU, and combine.')
    display(HTML(result_path.with_name('block_report.html').read_text()))
except Exception as error:
    failure = error
    details = traceback.format_exc()
    print(details)
    (RUNS / 'failure.txt').write_text(details)
finally:
    # A failure still exports source, environment output, and complete diagnostic logs.
    archive_path = PROJECT.parent / 'cuda_moe_engine_gpu_bundle.zip'
    excluded = {'__pycache__', '.pytest_cache', '.ruff_cache', '.git', 'build', 'dist'}
    with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for path in sorted(PROJECT.rglob('*')):
            relative = path.relative_to(PROJECT)
            if path.is_file() and not any(part in excluded or part.endswith('.egg-info')
                                          for part in relative.parts):
                archive.write(path, Path('cuda-moe-dispatch') / relative)
    print('Download this ZIP and send it back for review:', archive_path)
    display(FileLink(str(archive_path)))
    try:
        from google.colab import files
        files.download(str(archive_path))
    except ImportError:
        pass
if failure:
    raise RuntimeError('GPU validation stopped. Downloaded ZIP includes the complete error; send it back.') from failure


## After the run

- Open `runs/block-quick/block_report.html` locally for the portable demo.
- Raw samples, hardware, validation errors, and source hashes are in `block_results.json`.
- Open `runs/moe_block_trace.json` in a Chrome trace viewer for stage and kernel inspection.
- Send back the downloaded ZIP so the new GPU checks and full-forward measurements can be reviewed.

The project is synthetic inference engineering, with no training, multi-GPU communication,
pretrained-model evaluation, or production-library speed claim. Explicit persistent buffers are
reported separately from weights and backend temporary allocations. CUDA-event eager timings can
include CPU launch gaps. Stage medians are independent measurements and do not add up to forward.

[Grouped GEMM tutorial](https://triton-lang.org/main/getting-started/tutorials/08-grouped-gemm.html) ·
[PyTorch CUDA extensions](https://docs.pytorch.org/docs/stable/cpp_extension.html)
